# Эксперимент 15: новая логика сделки (entry/exit)

**Цель:** повторить быстрые тесты из `13_Complex_Models_And_Ensemble.ipynb` на тех же данных и сплите, но проверить альтернативную торговую логику:

- Вход в **long**: `p >= threshold`.
- Выход из long: `p < 0.5` (не ждём противоположный sell-сигнал).
- Вход в **short**: `p <= 1 - threshold`.
- Выход из short: `p > 0.5`.

Для честного сравнения считаем **две логики** на одинаковых прогнозах моделей:
1. `prod_hold` — текущая продовая (`hold_keep_prev=True`).
2. `entry_exit_05` — новая логика exit по 0.5.

Пороги уверенности: `20-80`, `25-75`, `30-70`, `35-65`, `40-60`.

Метрики: `auc_val`, `auc_test`, `net_%`, `trades`, `avg_%_per_trade`.

In [1]:
import sys, os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

HAS_XGB, HAS_LGB, HAS_CAT = False, False, False
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    pass
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    pass
try:
    import catboost as cb
    HAS_CAT = True
except ImportError:
    pass

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df.columns]

TARGET_COL = 'target'
sort_col = 'datetime' if 'datetime' in df.columns else 'timestamp'
COMMISSION_RT = 0.001
THR_PROD = 0.60
THRESHOLD_BANDS = {
    '20-80': 0.80,
    '25-75': 0.75,
    '30-70': 0.70,
    '35-65': 0.65,
    '40-60': 0.60,
}

valid = df.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
if len(dates) < 10:
    raise ValueError(f'Мало дат: {len(dates)}. Нужно минимум 10 дней.')
train_dates = set(dates[:8])
val_dates = set([dates[8]])
test_dates = set([dates[9]])

print('Temporal split:', f'train={min(train_dates)}..{max(train_dates)}', f'val={dates[8]}', f'test={dates[9]}')
print('Rows:', len(valid), '| BASE_FEATURES:', len(BASE_FEATURES))
print('XGB:', HAS_XGB, 'LGB:', HAS_LGB, 'CAT:', HAS_CAT)

Temporal split: train=2026-02-01..2026-02-08 val=2026-02-09 test=2026-02-10
Rows: 186063 | BASE_FEATURES: 22
XGB: True LGB: True CAT: True


In [2]:
def backtest_prod_hold(proba, ret, session_ids, threshold=THR_PROD, commission_rt=COMMISSION_RT, hold_keep_prev=True):
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev if hold_keep_prev else 0.0
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total = pos_changed.sum() * (commission_rt / 2.0)
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'net_%': float(pnl_net * 100), 'trades': trades, 'avg_%_per_trade': avg_trade}


def backtest_entry_exit_05(proba, ret, session_ids, threshold=THR_PROD, commission_rt=COMMISSION_RT):
    """Новая логика: входы по thr/1-thr, выходы по crossing 0.5."""
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0

        p = proba[i]
        if prev == 1.0:
            # Выход из long при p < 0.5
            if p < 0.5:
                prev = 0.0
        elif prev == -1.0:
            # Выход из short при p > 0.5
            if p > 0.5:
                prev = 0.0

        if prev == 0.0:
            if p >= threshold:
                prev = 1.0
            elif p <= 1 - threshold:
                prev = -1.0

        pos[i] = prev

    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total = pos_changed.sum() * (commission_rt / 2.0)
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'net_%': float(pnl_net * 100), 'trades': trades, 'avg_%_per_trade': avg_trade}


def prep(df_in, feat_cols):
    d = df_in.dropna(subset=feat_cols + [TARGET_COL]).copy()
    d = d[d[TARGET_COL].isin([-1.0, 1.0])]
    d['y'] = (d[TARGET_COL] == 1).astype(int)
    d['date'] = pd.to_datetime(d['datetime'], utc=True).dt.date
    d['ret_next'] = d.groupby('session_key')['close_price'].pct_change().shift(-1)
    d = d.dropna(subset=['ret_next'])
    tr = d[d['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    va = d[d['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    te = d[d['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    return tr, va, te, d


def fit_eval(name, model, feat_cols, tr, va, te):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(tr[feat_cols].fillna(0))
    Xva = scaler.transform(va[feat_cols].fillna(0))
    Xte = scaler.transform(te[feat_cols].fillna(0))
    ytr, yva, yte = tr['y'].values, va['y'].values, te['y'].values
    model.fit(Xtr, ytr)
    pva = model.predict_proba(Xva)[:, 1]
    pte = model.predict_proba(Xte)[:, 1]
    return {
        'model': name,
        'auc_val': roc_auc_score(yva, pva),
        'auc_test': roc_auc_score(yte, pte),
        'proba_test': pte,
    }


def build_trade_rows(base_result, te_df, feature_set):
    rows = []
    logic_map = {
        'prod_hold': backtest_prod_hold,
        'entry_exit_05': backtest_entry_exit_05,
    }
    for logic_name, bt_fn in logic_map.items():
        for band, thr in THRESHOLD_BANDS.items():
            bt = bt_fn(base_result['proba_test'], te_df['ret_next'].values, te_df['session_key'].values, threshold=thr)
            rows.append({
                'model': base_result['model'],
                'feature_set': feature_set,
                'logic': logic_name,
                'threshold_band': band,
                'threshold': thr,
                'auc_val': base_result['auc_val'],
                'auc_test': base_result['auc_test'],
                'net_%': bt['net_%'],
                'trades': bt['trades'],
                'avg_%_per_trade': bt['avg_%_per_trade'],
            })
    return rows


# Sequence features как в 13 (rolling 30/60) + расширенный вариант 5/15/30/60
key_feats = BASE_FEATURES[:10]
df_feat = df.copy()
grp = df_feat.groupby('session_key', group_keys=False)
for w in [5, 15, 30, 60]:
    for c in key_feats:
        if c in df_feat.columns:
            df_feat[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df_feat[f'{c}_roll{w}_std'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

old_cols = [c for c in df_feat.columns if ('_roll30_' in c or '_roll60_' in c)]
new_cols = [c for c in df_feat.columns if ('_roll5_' in c or '_roll15_' in c or '_roll30_' in c or '_roll60_' in c)]
FEATURES_BASE = [c for c in BASE_FEATURES if c in df_feat.columns]
FEATURES_OLD = [c for c in (BASE_FEATURES + old_cols) if c in df_feat.columns]
FEATURES_NEW = [c for c in (BASE_FEATURES + new_cols) if c in df_feat.columns]

tr_base, va_base, te_base, _ = prep(df_feat, FEATURES_BASE)
tr_old, va_old, te_old, all_old = prep(df_feat, FEATURES_OLD)
tr_new, va_new, te_new, all_new = prep(df_feat, FEATURES_NEW)
print('BASE features:', len(FEATURES_BASE), '| OLD:', len(FEATURES_OLD), '| NEW:', len(FEATURES_NEW))

BASE features: 22 | OLD: 62 | NEW: 102


In [3]:
def make_models():
    models = [
        ('logreg', LogisticRegression(max_iter=1200, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_split=8, min_samples_leaf=4, max_features='sqrt', random_state=42)),
    ]
    if HAS_XGB:
        models.append(('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')))
    if HAS_LGB:
        models.append(('lgb', lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)))
    if HAS_CAT:
        models.append(('cat', cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0)))
    return models

# Те же быстрые тесты, что в 13: baseline + sequence(30/60)
res_base, res_old = [], []
for n, m in make_models():
    base = fit_eval(f'{n}_base', m, FEATURES_BASE, tr_base, va_base, te_base)
    res_base.extend(build_trade_rows(base, te_base, 'baseline_22'))
for n, m in make_models():
    base = fit_eval(f'{n}_seq_old', m, FEATURES_OLD, tr_old, va_old, te_old)
    res_old.extend(build_trade_rows(base, te_old, 'sequence_30_60'))

# Дополнительно сравнение с расширенными окнами 5/15/30/60
res_new = []
for n, m in make_models():
    base = fit_eval(f'{n}_seq_new', m, FEATURES_NEW, tr_new, va_new, te_new)
    res_new.extend(build_trade_rows(base, te_new, 'sequence_5_15_30_60'))

# Прод-модель из 13 на тех же логиках
res_prod = []
prod_path = os.path.join(_root, 'models', 'best_model.joblib')
if os.path.exists(prod_path):
    bundle = joblib.load(prod_path)
    prod_model = bundle['model']
    prod_scaler = bundle['scaler']
    prod_feat = bundle['features']
    X_test_prod = prod_scaler.transform(te_base[prod_feat].fillna(0))
    pte = prod_model.predict_proba(X_test_prod)[:, 1]
    base_result = {
        'model': 'prod_catboost',
        'auc_val': np.nan,
        'auc_test': roc_auc_score(te_base['y'].values, pte),
        'proba_test': pte,
    }
    res_prod.extend(build_trade_rows(base_result, te_base, 'prod_baseline'))

all_rows = res_base + res_old + res_new + res_prod
summary = pd.DataFrame(all_rows)[['model', 'feature_set', 'logic', 'threshold_band', 'threshold', 'auc_val', 'auc_test', 'net_%', 'trades', 'avg_%_per_trade']]
summary = summary.sort_values(['feature_set', 'logic', 'threshold_band', 'net_%'], ascending=[True, True, True, False]).reset_index(drop=True)

print('=== Compare models and trading logic ===')
print(summary.to_string(index=False))

best = summary.sort_values('net_%', ascending=False).iloc[0]
print('\nBest by net_%:', best['model'], '|', best['feature_set'], '|', best['logic'], '|', best['threshold_band'], '|', f"net={best['net_%']:.2f}%", '|', f"avg/trade={best['avg_%_per_trade']:.4f}%")

=== Compare models and trading logic ===
         model         feature_set         logic threshold_band  threshold  auc_val  auc_test       net_%  trades  avg_%_per_trade
      xgb_base         baseline_22 entry_exit_05          20-80       0.80 0.753120  0.623011  475.383063     923         0.515041
      lgb_base         baseline_22 entry_exit_05          20-80       0.80 0.755425  0.628289  363.555387     689         0.527657
      cat_base         baseline_22 entry_exit_05          20-80       0.80 0.757723  0.626608  327.405342     423         0.774008
       rf_base         baseline_22 entry_exit_05          20-80       0.80 0.762590  0.617960  165.308219     260         0.635801
   logreg_base         baseline_22 entry_exit_05          20-80       0.80 0.722027  0.613062   75.183165      83         0.905821
      xgb_base         baseline_22 entry_exit_05          25-75       0.75 0.753120  0.623011  586.659431    1656         0.354263
      lgb_base         baseline_22 entry_e

In [4]:
if HAS_LGB:
    eval_days = [d for d in dates if d not in train_dates]

    def fit_lgb_daywise(feat_cols, variant_label):
        d = df_feat.dropna(subset=feat_cols + [TARGET_COL]).copy()
        d = d[d[TARGET_COL].isin([-1.0, 1.0])]
        d['y'] = (d[TARGET_COL] == 1).astype(int)
        d['date'] = pd.to_datetime(d['datetime'], utc=True).dt.date
        d['ret_next'] = d.groupby('session_key')['close_price'].pct_change().shift(-1)
        d = d.dropna(subset=['ret_next'])

        tr = d[d['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(tr[feat_cols].fillna(0))
        ytr = tr['y'].values
        model = lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)
        model.fit(Xtr, ytr)

        rows = []
        for day in eval_days:
            dd = d[d['date'] == day].sort_values(['session_key', sort_col]).reset_index(drop=True)
            if len(dd) == 0:
                continue
            Xd = scaler.transform(dd[feat_cols].fillna(0))
            p = model.predict_proba(Xd)[:, 1]
            auc = roc_auc_score(dd['y'].values, p)
            for band, thr in THRESHOLD_BANDS.items():
                bt_prod = backtest_prod_hold(p, dd['ret_next'].values, dd['session_key'].values, threshold=thr)
                bt_new = backtest_entry_exit_05(p, dd['ret_next'].values, dd['session_key'].values, threshold=thr)
                rows.append({'day': day, 'variant': variant_label, 'logic': 'prod_hold', 'threshold_band': band, 'threshold': thr, 'auc': auc, 'net_%': bt_prod['net_%'], 'trades': bt_prod['trades'], 'avg_%_per_trade': bt_prod['avg_%_per_trade']})
                rows.append({'day': day, 'variant': variant_label, 'logic': 'entry_exit_05', 'threshold_band': band, 'threshold': thr, 'auc': auc, 'net_%': bt_new['net_%'], 'trades': bt_new['trades'], 'avg_%_per_trade': bt_new['avg_%_per_trade']})
        return rows

    stab = pd.DataFrame(
        fit_lgb_daywise(FEATURES_OLD, 'sequence_30_60') +
        fit_lgb_daywise(FEATURES_NEW, 'sequence_5_15_30_60')
    )
    stab = stab.sort_values(['day', 'variant', 'logic', 'threshold_band']).reset_index(drop=True)
    print('=== Day-by-day stability (lgb_seq, both trading logics) ===')
    print(stab.to_string(index=False))

    agg = stab.groupby(['variant', 'logic', 'threshold_band'], as_index=False).agg(
        days=('day', 'count'),
        auc_mean=('auc', 'mean'),
        net_mean=('net_%', 'mean'),
        net_median=('net_%', 'median'),
        trades_mean=('trades', 'mean'),
        avg_trade_mean=('avg_%_per_trade', 'mean')
    )
    print('\n=== Stability aggregate ===')
    print(agg.to_string(index=False))
else:
    print('LightGBM не установлен, блок стабильности пропущен.')

=== Day-by-day stability (lgb_seq, both trading logics) ===
       day             variant         logic threshold_band  threshold      auc       net_%  trades  avg_%_per_trade
2026-02-09      sequence_30_60 entry_exit_05          20-80       0.80 0.773493 1078.646340    1499         0.719577
2026-02-09      sequence_30_60 entry_exit_05          25-75       0.75 0.773493 1550.268062    2458         0.630703
2026-02-09      sequence_30_60 entry_exit_05          30-70       0.70 0.773493 1944.168626    3591         0.541400
2026-02-09      sequence_30_60 entry_exit_05          35-65       0.65 0.773493 2300.132190    4835         0.475725
2026-02-09      sequence_30_60 entry_exit_05          40-60       0.60 0.773493 2578.065277    6121         0.421184
2026-02-09      sequence_30_60     prod_hold          20-80       0.80 0.773493 1059.562037     453         2.338989
2026-02-09      sequence_30_60     prod_hold          25-75       0.75 0.773493 1490.900031     733         2.033970
2026

## Итоговые выводы по ноутбуку

### Сравнение двух логик сделок

1. **`entry_exit_05` увеличивает частоту сделок в 2–4 раза** по сравнению с `prod_hold` при одинаковом пороге. Это повышает net_% в бектесте, но ухудшает устойчивость к slippage.

2. **`prod_hold` даёт заметно выше `avg_%_per_trade`.** На 2026-02-09 при 40-60: `prod_hold` ~1.10% на сделку vs `entry_exit_05` ~0.42%. Новая логика «выжимает» PnL количеством, а не качеством.

3. **Расширенные окна `5/15/30/60` лучше `30/60`:** рост AUC (0.793 vs 0.773), выше net_% в большинстве порогов.

4. **Рабочая зона порогов: `25-75` / `30-70`.** 20-80 — мало сделок; 40-60 — слишком много.

### Лучшие конфигурации для прода (avg/trade ≥ 2%, trades 300–800)

Из day-by-day stability (модель — **LightGBM**, обучение на 8 днях, оценка на 2 днях):

| Модель | Feature set | Logic | Band | День | net_% | trades | avg/trade |
|--------|------------|-------|------|------|-------|--------|-----------|
| **LightGBM** | seq_5_15_30_60 | prod_hold | 20-80 | 02-09 | 1128% | 488 | **2.31%** |
| **LightGBM** | seq_5_15_30_60 | prod_hold | 25-75 | 02-09 | 1657% | 793 | **2.09%** |
| **LightGBM** | sequence_30_60 | prod_hold | 20-80 | 02-09 | 1060% | 453 | **2.34%** |
| **LightGBM** | sequence_30_60 | prod_hold | 25-75 | 02-09 | 1491% | 733 | **2.03%** |

Средние по 2 дням (Stability aggregate):

| Модель | Feature set | Logic | Band | net_mean | trades_mean | avg_trade_mean |
|--------|------------|-------|------|----------|-------------|----------------|
| **LightGBM** | seq_5_15_30_60 | prod_hold | 20-80 | 761% | 343 | **2.16%** |
| **LightGBM** | seq_5_15_30_60 | prod_hold | 25-75 | 1117% | 555 | **1.96%** |
| **LightGBM** | sequence_30_60 | prod_hold | 20-80 | 706% | 314 | **2.18%** |
| **LightGBM** | sequence_30_60 | prod_hold | 25-75 | 981% | 524 | **1.77%** |

### Почему LightGBM показал себя лучше других

1. **Калибровка вероятностей.** LGB выдаёт хорошо распределённые вероятности — при пороге 20-80 проходит достаточно сигналов (300–500/день), а не 14–51 как у logreg/rf. Это даёт статистически значимое количество сделок с высоким avg/trade.

2. **Устойчивость к шуму rolling-фичей.** XGBoost при тех же фичах (seq_5_15_30_60) даёт AUC_val 0.791, но LGB (0.793) стабильнее генерализуется: на test-дне LGB сохраняет avg/trade > 1.2%, тогда как XGB просаживается сильнее.

3. **Баланс сложности.** CatBoost при 20-80 генерирует ещё меньше сделок (128–239), а logreg/rf — слишком мало или слишком неточно. LGB попадает в оптимум: достаточно сложный для хорошего AUC, достаточно «спокойный» для не-переобученных вероятностей.

### Конфигурации с avg/trade ≥ 2%, но мало сделок (статистически ненадёжные)

| Модель | Feature set | Logic | Band | net_% | trades | avg/trade | Проблема |
|--------|------------|-------|------|-------|--------|-----------|----------|
| logreg_base | baseline_22 | prod_hold | 20-80 | 129% | 37 | 3.48% | 37 сделок — случайность |
| logreg_seq_old | seq_30_60 | prod_hold | 20-80 | 174% | 51 | 3.41% | мало сделок |
| rf_seq_old | seq_30_60 | prod_hold | 20-80 | 42% | 14 | 3.00% | 14 сделок — шум |
| logreg_seq_new | seq_5_15_30_60 | prod_hold | 20-80 | 226% | 76 | 2.98% | мало сделок |

Эти конфигурации интересны как индикатор (logreg/rf при 20-80 фильтруют только самые «очевидные» моменты), но **не пригодны для прода** из-за малого числа сделок и нестабильной статистики.

### Общие паттерны

1. **Все лидеры — `prod_hold`.** Ни одной `entry_exit_05`. Hold-логика захватывает длинные движения → каждая сделка «жирнее».
2. **Все лидеры — строгие пороги `20-80` или `25-75`.** Строгий фильтр = мало ложных входов.
3. **`seq_5_15_30_60` > `seq_30_60`.** Короткие окна (5, 15 мин) добавляют локальный сигнал → рост AUC и net_% при сопоставимом avg/trade.
4. **Val-день (02-09) лучше test-дня (02-10).** Разрыв avg/trade ~40%. Нужна проверка на большем количестве дней.

### Практический вывод

**Лучший кандидат для paper/live-теста: LightGBM + seq_5_15_30_60 + prod_hold + порог 25-75.**

- avg/trade ~2.0% (среднее по 2 дням ~1.96%) — огромный запас на slippage.
- ~555 сделок/день — достаточно для стабильной статистики.
- net_mean ~1117% — даже при 50% деградации в реале это отличный результат.

Альтернатива: порог **20-80** (avg/trade 2.16%, но ~343 сделки — на нижней границе комфорта).

`entry_exit_05` — оставить как альтернативный режим после моделирования slippage и лимита по числу сделок.